# Entrenamiento de modelos clásicos para calidad de frutas

Este notebook entrena una línea base, un SVM lineal y un Random Forest con el split compartido, 69 características y la misma prueba final. La búsqueda de hiperparámetros usa validación cruzada únicamente dentro de `train`; `test` queda reservado para la evaluación final.

## Cómo entran y se evalúan las fotos propias

`load_shared_dataset()` (sección 2, celda de código) cruza `features_all.csv` con `dataset_splits.csv` usando fuente, clase, fruta y nombre de archivo. Las filas `source_origin=own` se mapean a la fuente `own`. Ahora el inventario contiene 50 fotos propias: 36 iniciales y 14 nuevas. Las seis fotos que se usaron como prueba en la ejecución anterior ya se habían inspeccionado, así que para esta ejecución pasan a `train`. El split actual asigna 36 propias a `train`, 8 a `val` y reserva las 6 nuevas para `test` (2 por clase).

La sección 3 entrena con `split == "train"` y evalúa al final con `split == "test"`. Las 8 fotos de `val` quedan reservadas; este entrenamiento ajusta hiperparámetros mediante validación cruzada dentro de `train`. En la función `_save_result` (sección 2), cada modelo calcula métricas generales y métricas por fuente. El resultado específico de las seis fotos propias nuevas queda en `results["models"][modelo]["by_source"]["own"]` dentro de `results/metrics/classical_models_test.json`; las predicciones individuales incluyen la columna `source` en los CSV de `results/predictions/`.

La evaluación propia tiene solo 6 casos, así que cada acierto cambia la exactitud en 16,7 puntos porcentuales. Es una comprobación inicial con fotos nuevas del grupo, pero sus métricas tienen alta incertidumbre y deben interpretarse con cautela.

## 1. Imports, rutas y configuración

In [1]:
"""Train and evaluate a baseline, linear SVM, and Random Forest on the shared split.
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

ROOT = Path.cwd().resolve()
while not (ROOT / 'requirements.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'requirements.txt').exists():
    raise RuntimeError('Abre o ejecuta el notebook desde la raíz del repositorio.')
sys.path.insert(0, str(ROOT))
from src.utils.config import CLASSES, SEED  # noqa: E402
from src.utils.evaluation import save_predictions  # noqa: E402

FEATURES_PATH = ROOT / "src" / "data" / "processed" / "features_all.csv"
SPLITS_PATH = ROOT / "src" / "data" / "splits" / "dataset_splits.csv"
MODEL_DIR = ROOT / "models_saved"
PREDICTIONS_DIR = ROOT / "results" / "predictions"
METRICS_PATH = ROOT / "results" / "metrics" / "classical_models_test.json"
LABEL_NAMES = {0: CLASSES[0], 1: CLASSES[1], 2: CLASSES[2]}
METADATA_COLUMNS = {"rel_path", "source", "class", "label", "fruit"}




## 2. Cruce de características con el split 

In [2]:
def _normal_key(source: pd.Series, class_name: pd.Series,
                fruit: pd.Series, basename: pd.Series) -> pd.Series:
    return (
        source.astype(str).str.lower() + "|"
        + class_name.astype(str).str.lower() + "|"
        + fruit.astype(str).str.lower() + "|"
        + basename.astype(str).str.lower()
    )


def load_shared_dataset():
    if not FEATURES_PATH.exists() or not SPLITS_PATH.exists():
        raise FileNotFoundError(
            f"Se requieren {FEATURES_PATH.relative_to(ROOT)} y "
            f"{SPLITS_PATH.relative_to(ROOT)}."
        )

    features = pd.read_csv(FEATURES_PATH)
    split = pd.read_csv(SPLITS_PATH)
    required_features = METADATA_COLUMNS | {"diameter_norm"}
    required_split = {
        "class", "label", "source_origin", "fruit_subfolder",
        "source_path", "split", "is_augmented",
    }
    if missing := required_features - set(features.columns):
        raise ValueError(f"Faltan columnas en features_all.csv: {sorted(missing)}")
    if missing := required_split - set(split.columns):
        raise ValueError(f"Faltan columnas en dataset_splits.csv: {sorted(missing)}")

    # Keep original images. Augmented rows point back to their source image and
    # would otherwise duplicate a single feature vector inside training.
    is_augmented = split["is_augmented"].astype(str).str.lower().isin({"true", "1"})
    split = split.loc[~is_augmented].copy()

    source_map = {"kaggle": "kaggle", "local_apo3": "onedrive", "own": "own"}
    split_source = split["source_origin"].map(source_map)
    if split_source.isna().any():
        unknown = sorted(split.loc[split_source.isna(), "source_origin"].astype(str).unique())
        raise ValueError(f"Fuentes del split sin mapeo: {unknown}")

    features["_basename"] = features["rel_path"].astype(str).str.split("/").str[-1]
    source_path = split["source_path"].astype(str).str.replace(chr(92), "/", regex=False)
    split["_basename"] = source_path.str.split("/").str[-1]
    split_fruit = split["fruit_subfolder"].astype(str).str.split("_").str[0]

    features["_key"] = _normal_key(
        features["source"], features["class"], features["fruit"], features["_basename"]
    )
    split["_key"] = _normal_key(
        split_source, split["class"], split_fruit, split["_basename"]
    )
    if features["_key"].duplicated().any() or split["_key"].duplicated().any():
        raise ValueError("Hay claves duplicadas; se cancela para evitar asignar un split incorrecto.")

    split_rows = split[["_key", "split", "label"]].rename(columns={"label": "split_label"})
    joined = features.merge(split_rows, on="_key", how="inner", validate="one_to_one")
    joined["label"] = pd.to_numeric(joined["label"], errors="raise").astype(int)
    joined["split_label"] = pd.to_numeric(joined["split_label"], errors="raise").astype(int)
    if not joined["label"].equals(joined["split_label"]):
        raise ValueError("Las etiquetas de features_all.csv no coinciden con las del split.")
    if set(joined["label"].unique()) != set(LABEL_NAMES):
        raise ValueError(f"Se esperaban las etiquetas 0, 1 y 2; encontré {sorted(joined['label'].unique())}.")

    feature_columns = [c for c in features.columns if c not in METADATA_COLUMNS | {"_basename", "_key"}]
    if len(feature_columns) != 69:
        raise ValueError(f"Se esperaban 69 características, encontré {len(feature_columns)}.")
    if not np.isfinite(joined[feature_columns].to_numpy(dtype=np.float64)).all():
        raise ValueError("Hay NaN o valores infinitos en las características.")

    joined["label_name"] = joined["label"].map(LABEL_NAMES)
    diagnostics = {
        "feature_rows": int(len(features)),
        "original_split_rows": int(len(split)),
        "matched_rows": int(len(joined)),
        "split_rows_without_features": int(len(split) - len(joined)),
        "feature_rows_without_split": int(len(features) - len(joined)),
        "unmatched_split_by_class_source": {
            f"{source}|{class_name}": int(count)
            for (source, class_name), count in (
                split.loc[~split["_key"].isin(set(features["_key"]))]
                .groupby(["source_origin", "class"]).size().items()
            )
        },
        "unmatched_features_by_source_class": {
            f"{source}|{class_name}": int(count)
            for (source, class_name), count in (
                features.loc[~features["_key"].isin(set(split["_key"]))]
                .groupby(["source", "class"]).size().items()
            )
        },
        "matched_by_split": {
            str(name): int(count) for name, count in joined.groupby("split").size().items()
        },
    }
    return joined, feature_columns, diagnostics


def _metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_names = pd.Series(y_true).map(LABEL_NAMES).to_numpy()
    pred_names = pd.Series(y_pred).map(LABEL_NAMES).to_numpy()
    supported = [name for name in CLASSES if np.any(true_names == name)]
    per_class = {}
    recalls = []
    f1_values = []
    for name in supported:
        tp = int(np.sum((true_names == name) & (pred_names == name)))
        fp = int(np.sum((true_names != name) & (pred_names == name)))
        fn = int(np.sum((true_names == name) & (pred_names != name)))
        support = int(np.sum(true_names == name))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        per_class[name] = {
            "precision": precision, "recall": recall, "f1": f1, "support": support,
        }
        recalls.append(recall)
        f1_values.append(f1)
    return {
        "accuracy": float(accuracy_score(true_names, pred_names)),
        "balanced_accuracy": float(np.mean(recalls)),
        "f1_macro": float(np.mean(f1_values)),
        "classes_with_test_examples": supported,
        "per_class": per_class,
        "confusion_matrix_class_order": list(CLASSES),
        "confusion_matrix": confusion_matrix(true_names, pred_names, labels=CLASSES).tolist(),
    }


def _save_result(name: str, estimator, X_test: pd.DataFrame,
                 test_rows: pd.DataFrame) -> dict:
    y_true = test_rows["label"].to_numpy(dtype=int)
    y_pred = estimator.predict(X_test).astype(int)
    probabilities = estimator.predict_proba(X_test)
    estimator_classes = list(estimator.classes_)
    order = [estimator_classes.index(label) for label in LABEL_NAMES]
    probabilities = probabilities[:, order]

    prediction_path = save_predictions(
        paths=test_rows["rel_path"].astype(str).tolist(),
        y_true=pd.Series(y_true).map(LABEL_NAMES).to_numpy(),
        y_pred=pd.Series(y_pred).map(LABEL_NAMES).to_numpy(),
        y_proba=probabilities,
        model_name=name,
        split="test",
    )
    prediction_df = pd.read_csv(prediction_path)
    prediction_df.insert(1, "source", test_rows["source"].to_numpy())
    prediction_df.to_csv(prediction_path, index=False)

    by_source = {}
    for source, indices in test_rows.groupby("source").groups.items():
        positions = np.asarray(list(indices), dtype=int)
        by_source[source] = _metrics(y_true[positions], y_pred[positions])

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(estimator, MODEL_DIR / f"{name}.joblib")
    return {
        "test_rows": int(len(test_rows)),
        "overall": _metrics(y_true, y_pred),
        "by_source": by_source,
        "predictions_file": str(prediction_path.relative_to(ROOT)),
        "model_file": str((MODEL_DIR / f"{name}.joblib").relative_to(ROOT)),
    }



## 3. Preparar train, test y métricas

In [3]:
data, feature_columns, diagnostics = load_shared_dataset()
train = data[data["split"] == "train"].reset_index(drop=True)
test = data[data["split"] == "test"].reset_index(drop=True)
if train.empty or test.empty:
    raise ValueError("El split debe contener filas tanto de train como de test.")

X_train, y_train = train[feature_columns], train["label"].to_numpy(dtype=int)
X_test = test[feature_columns]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
results = {
    "data_join": diagnostics,
    "feature_count": len(feature_columns),
    "label_order": LABEL_NAMES,
    "cv": {"type": "StratifiedKFold", "n_splits": 3, "seed": SEED,
           "scoring": "f1_macro"},
    "models": {},
}
print(f"Train: {len(train)} | Test: {len(test)} | Características: {len(feature_columns)}", flush=True)
print(f"Join: {diagnostics['matched_rows']} coincidencias; "
      f"{diagnostics['split_rows_without_features']} filas del split sin características.", flush=True)



Train: 19243 | Test: 4112 | Características: 69


Join: 27465 coincidencias; 543 filas del split sin características.


## 4. Línea base por clase mayoritaria

In [4]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
results["models"]["baseline"] = _save_result("baseline", baseline, X_test, test)
print("Línea base lista.", flush=True)



Línea base lista.


## 5. SVM lineal

In [5]:
svm_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("svm", LinearSVC(class_weight="balanced", random_state=SEED,
                       max_iter=20000, dual="auto")),
])
svm_search = GridSearchCV(
    svm_pipeline, {"svm__C": [0.01, 0.1, 1.0, 10.0]},
    scoring="f1_macro", cv=cv, refit=True, n_jobs=1, verbose=2,
)
print("Buscando hiperparámetros del SVM (4 configuraciones × 3 folds)...", flush=True)
svm_search.fit(X_train, y_train)
calibrated_svm = CalibratedClassifierCV(svm_search.best_estimator_, method="sigmoid", cv=3)
calibrated_svm.fit(X_train, y_train)
results["models"]["svm_linear"] = _save_result("svm_linear", calibrated_svm, X_test, test)
results["models"]["svm_linear"]["best_params"] = svm_search.best_params_
results["models"]["svm_linear"]["best_cv_f1_macro"] = float(svm_search.best_score_)
print(f"SVM listo. Mejor CV F1 macro: {svm_search.best_score_:.4f}", flush=True)



Buscando hiperparámetros del SVM (4 configuraciones × 3 folds)...


Fitting 3 folds for each of 4 candidates, totalling 12 fits


[CV] END ........................................svm__C=0.01; total time=   0.5s


[CV] END ........................................svm__C=0.01; total time=   0.4s


[CV] END ........................................svm__C=0.01; total time=   0.3s


[CV] END .........................................svm__C=0.1; total time=   0.6s


... (5 outputs left out)

[CV] END ........................................svm__C=10.0; total time=   1.9s


[CV] END ........................................svm__C=10.0; total time=   2.2s


[CV] END ........................................svm__C=10.0; total time=   2.5s


SVM listo. Mejor CV F1 macro: 0.8641


## 6. Random Forest: búsqueda de hiperparámetros

In [6]:
forest = RandomForestClassifier(
    class_weight="balanced_subsample", random_state=SEED, n_jobs=-1,
)
forest_search = GridSearchCV(
    forest,
    {
        "n_estimators": [100, 200],
        "max_depth": [None, 24],
        "min_samples_leaf": [1, 3],
    },
    scoring="f1_macro", cv=cv, refit=True, n_jobs=1, verbose=2,
)
print("Buscando hiperparámetros del Random Forest (8 configuraciones × 3 folds)...", flush=True)
forest_search.fit(X_train, y_train)
results["models"]["random_forest"] = _save_result(
    "random_forest", forest_search.best_estimator_, X_test, test,
)
results["models"]["random_forest"]["best_params"] = forest_search.best_params_
results["models"]["random_forest"]["best_cv_f1_macro"] = float(forest_search.best_score_)
print(f"Random Forest listo. Mejor CV F1 macro: {forest_search.best_score_:.4f}", flush=True)



Buscando hiperparámetros del Random Forest (8 configuraciones × 3 folds)...


Fitting 3 folds for each of 8 candidates, totalling 24 fits


[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   1.7s


[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   1.6s


[CV] END max_depth=None, min_samples_leaf=1, n_estimators=100; total time=   1.6s


[CV] END max_depth=None, min_samples_leaf=1, n_estimators=200; total time=   3.1s


... (17 outputs left out)

[CV] END .max_depth=24, min_samples_leaf=3, n_estimators=200; total time=   2.9s


[CV] END .max_depth=24, min_samples_leaf=3, n_estimators=200; total time=   2.9s


[CV] END .max_depth=24, min_samples_leaf=3, n_estimators=200; total time=   2.9s


Random Forest listo. Mejor CV F1 macro: 0.9367


## 7. Guardar métricas y comparar los modelos

In [7]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Métricas y matrices guardadas en {METRICS_PATH.relative_to(ROOT)}", flush=True)
for name, result in results["models"].items():
    overall = result["overall"]
    print(f"{name}: accuracy={overall['accuracy']:.4f}, "
          f"balanced_accuracy={overall['balanced_accuracy']:.4f}, "
          f"F1_macro={overall['f1_macro']:.4f}", flush=True)


Métricas y matrices guardadas en results\metrics\classical_models_test.json


baseline: accuracy=0.5229, balanced_accuracy=0.3333, F1_macro=0.2289


svm_linear: accuracy=0.8833, balanced_accuracy=0.8504, F1_macro=0.8658


random_forest: accuracy=0.9560, balanced_accuracy=0.9522, F1_macro=0.9445
